# Task 1 — Generating Acceptance Tests for Dependability Properties

In this notebook we turn the *improved* user stories of the Traffic Light System into **acceptance tests (ATs)** that serve to verify the system's dependability.

We will use the four dependability properties defined in the Verification and Reliability course:
- **Correctness** — does exactly what the spec says
- **Reliability** — keeps doing it correctly over time
- **Robustness** — copes with bad/unexpected input without breaking
- **Safety** — never causes a disaster (here: two crossing directions must never both be Green)

**Pipeline:** (1) parse the improved stories → (2) build a focused prompt per story including its dependability label → (3) send each story individually to a local LLM (Ollama / llama3) → (4) collect the AT tables and time the run → (5) save everything with metadata to `artifacts/acceptance_tests_output.txt`.

Each generated AT will be labelled by the property it verifies and contains at least one **valid** and one **invalid** case.

## Setup

We talk to **Ollama**, which runs an LLM locally. 

Starting with `ollama serve` command from a terminal -> it starts listening at `http://localhost:11434`.

In the code we POST a prompt to `/api/generate` and read the response.

- `MODEL_NAME = "llama3"` — the local model that writes the AT tables.
- `ARTIFACTS_DIR` — keeps the input stories and output ATs together so the task folder is self-contained in the repo.

Libraries used in this setup:
- `requests` — HTTP calls to Ollama
- `time` — measure generation time (the assignment asks us to report it)
- `re` — regex to parse story names and dependability labels
- `pathlib` — clean cross-platform file paths

In [ ]:
import requests
import time
import re
from pathlib import Path

# Ollama config
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3"

# Pathing
ARTIFACTS_DIR = Path("artifacts")
STORIES_FILE = ARTIFACTS_DIR / "UserStories_improved.txt"
OUTPUT_FILE = ARTIFACTS_DIR / "acceptance_tests_output.txt"

print(f"Stories file: {STORIES_FILE}")
print(f"Output file: {OUTPUT_FILE}")

## Step 1 — Parse the improved user stories

Rather than sending the whole file as one blob (like the previously provided `acceptance_test_generator.py` example does), we split it into eight individual stories. Decision was made with the best prompt engineering techniques in mind. Ollama will perform better when stories are sent individually as it can focus more on each story.

Each block starts with a line beginning `US-name`, so we accumulate lines until the next `US-name` marker.

For each story we extract with regex:
- the **name** (e.g. `executeCycle`)
- the **dependability label** (ex: `Reliability`) from the `[...]` tag

In [ ]:
# read the stories file
with open(STORIES_FILE, "r") as f:
    content = f.read()

# split by story lines (US-name : ...)
# simple regex split on lines starting with "US-name"
stories = []
current = []

for line in content.split('\n'):
    if line.startswith('US-name'):
        if current:
            stories.append('\n'.join(current))
        current = [line]
    else:
        current.append(line)

if current:
    stories.append('\n'.join(current))

# extract names and labels
story_info = []
for story in stories:
    # extract US-name and label -> like correctness or safety etc
    name_match = re.search(r'US-name\s*:\s*(\w+)', story)
    label_match = re.search(r'\[(\w+)\]', story)
    
    if name_match and label_match:
        name = name_match.group(1)
        label = label_match.group(1)
        story_info.append({'name': name, 'label': label, 'text': story})

print(f"Found {len(story_info)} stories:")
for info in story_info:
    print(f"  - {info['name']} [{info['label']}]")

## Step 2 — Prompt template

The template formats every answer, so all eight ATs come out in a uniform format. It instructs the model to:

1. Produce a single AT table headed by the AT name and dependability property
2. Include both a **valid case** (what the system should do) and an **invalid case** (what the system should refuse) and the invalid case is where safety and robustness are really tested
3. Output only the table, no surrounding prose

Fixing the structure up front will keep the saved output file consistent and carries the dependability label directly into the table header.

In [ ]:
TEMPLATE_PROMPT = """You are a software requirements engineer.

Using the following template, create an Acceptance Test table
for the given User Story. Label it with the dependability property.

Use EXACTLY this structure:

| AT name: <testName> | [<DependabilityProperty>] | |
| ---- | ---- | ---- |
| Input | Description | Expected behavior |
| ... | ... | ... |
| ... | ... | ... |
| | | |
| Invalid condition | Description | Expected behavior |
| US name: <UserStoryName> | | |

Rules:
- ONE Acceptance Test for this story
- Include valid case(s)
- Include invalid case
- Keep the format identical
- Do NOT explain anything
- Output only the table
"""

print("Template ready.")

## Step 3 — Generate acceptance tests via Ollama

`generate_at_for_story(...)` appends the story text and its dependability label to the template prompt, POSTs it to Ollama, and returns the raw response.

A `timeout` plus `try/except` stops one slow or failed call from killing the entire run.

In [ ]:
def generate_at_for_story(story_text, story_name, dependability_label):
    """Send one story to Ollama, get back an AT table."""
    prompt = TEMPLATE_PROMPT + f"""

Dependability property: {dependability_label}

User Story:
{story_text}
"""
    
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False
            },
            timeout=60
        )
        response.raise_for_status()
        result = response.json()["response"]
        return result
    except Exception as e:
        return f"ERROR: {str(e)}"

print("Generator function ready.")

In [ ]:
# Generate ATs for all stories
print(f"Generating {len(story_info)} acceptance tests...\n")
start_time = time.time()

all_ats = []
for i, info in enumerate(story_info, 1):
    print(f"[{i}/{len(story_info)}] Generating AT for {info['name']} [{info['label']}]...")
    
    at_result = generate_at_for_story(
        info['text'],
        info['name'],
        info['label']
    )
    
    all_ats.append({
        'story_name': info['name'],
        'label': info['label'],
        'at_table': at_result
    })
    
    print(f"  ✓ Generated ({len(at_result)} chars)\n")

end_time = time.time()
elapsed = end_time - start_time

print(f"\n{'='*60}")
print(f"Generation complete in {elapsed:.1f} seconds")
print(f"Average time per story: {elapsed/len(story_info):.1f} seconds")
print(f"{'='*60}")

## Step 4 — Save all ATs to the output file

We write all eight AT tables to `artifacts/acceptance_tests_output.txt`. The file header will record:
- Timestamp of the run
- Total generation time
- Model used

This makes the run auditable and roughly reproducible. 

Each AT section is headed by its story name and dependability property, so the file is easy to navigate.

In [ ]:
# write to output file
with open(OUTPUT_FILE, "w") as f:
    f.write("ACCEPTANCE TESTS (Task 1)\n")
    f.write("="*70 + "\n")
    f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Total generation time: {elapsed:.1f} seconds\n")
    f.write(f"Model: {MODEL_NAME}\n")
    f.write("="*70 + "\n\n")
    
    for i, at_info in enumerate(all_ats, 1):
        f.write(f"\n[{i}] Story: {at_info['story_name']}  |  Dependability: {at_info['label']}\n")
        f.write("-"*70 + "\n")
        f.write(at_info['at_table'])
        f.write("\n\n")

print(f"✓ All ATs saved to: {OUTPUT_FILE}")
print(f"\nYou can now find the acceptance tests in artifacts/acceptance_tests_output.txt")

## Step 5 — Preview
We should print the first generated AT as a quick sanity check, confirm the format looks correct before opening the full output file.

In [ ]:
# show first AT as a preview
print("PREVIEW: First Acceptance Test\n")
print(all_ats[0]['at_table'])